In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD NOTEBOOKS 62/63'S REAL POLICY AND
#            RESULTS
# =============================================================================
import os
import sys
import gc
import json
import time
import importlib.util
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Notebooks 62/63's Real Policy and Results")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB50_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_50_summary.json"
NB62_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_62_summary.json"
NB63_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_63_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB50_SUMMARY_PATH, "run 50_collections_optimization_business_understanding.ipynb first (Problem 9)"),
    (NB62_SUMMARY_PATH, "run 62_customer_intelligence_business_understanding.ipynb first (Problem 12)"),
    (NB63_SUMMARY_PATH, "run 63_customer_intelligence_modeling.ipynb first (Problem 12)"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB50_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB50_SUMMARY = json.load(f)
with open(NB62_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB62_SUMMARY = json.load(f)
with open(NB63_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB63_SUMMARY = json.load(f)

POLICY_PATH = Path(NB62_SUMMARY["policy_path"])
with open(POLICY_PATH, "r", encoding="utf-8") as f:
    CUSTOMER_INTELLIGENCE_POLICY = json.load(f)
MODELING_RESULTS_PATH = Path(NB63_SUMMARY["modeling_results_path"])
with open(MODELING_RESULTS_PATH, "r", encoding="utf-8") as f:
    MODELING_RESULTS = json.load(f)
PROFILE_PATH = Path(NB63_SUMMARY["profile_path"])
if not PROFILE_PATH.exists():
    raise FileNotFoundError(f"{PROFILE_PATH} not found.\nFix: re-run Notebook 63.")

UNIFIED_SCORE_WEIGHTS = CUSTOMER_INTELLIGENCE_POLICY["unified_score_weights"]
UNIFIED_RISK_GRADE_NAMES = CUSTOMER_INTELLIGENCE_POLICY["unified_risk_grade_names"]
UNIFIED_RISK_GRADE_CUT_PERCENTILES = CUSTOMER_INTELLIGENCE_POLICY["unified_risk_grade_cut_percentiles"]
KPI_TARGETS = CUSTOMER_INTELLIGENCE_POLICY["kpi_targets"]

REPORTED_UNIFIED_ROC_AUC = MODELING_RESULTS["kpi_results"]["composite_non_inferiority"]["unified_roc_auc"]
REPORTED_STATIC_PD_AUC = MODELING_RESULTS["kpi_results"]["composite_non_inferiority"]["static_pd_roc_auc"]
REPORTED_DYNAMIC_PD_AUC = MODELING_RESULTS["kpi_results"]["composite_non_inferiority"]["dynamic_pd_roc_auc"]
REPORTED_BEST_SINGLE_SIGNAL_AUC = MODELING_RESULTS["kpi_results"]["composite_non_inferiority"]["best_single_signal_auc"]
REPORTED_COMPOSITE_TOLERANCE = MODELING_RESULTS["kpi_results"]["composite_non_inferiority"]["tolerance"]
REPORTED_GRADE_CUT_LOW = MODELING_RESULTS["unified_grade_cut_low"]
REPORTED_GRADE_CUT_HIGH = MODELING_RESULTS["unified_grade_cut_high"]
REPORTED_RECOMMENDED_FOR_PRODUCTION = MODELING_RESULTS["recommended_for_production"]

# --- Problem 9's real persisted propensity-to-cure model + its own real
#     validated deployment policy -- every value below is read from
#     Notebook 62's own recorded paths, never re-derived or guessed. ---
P9_REUSE = CUSTOMER_INTELLIGENCE_POLICY["reused_from_problem_9"]
P9_MODEL_PATH = Path(P9_REUSE["model_path"])
P9_DEPLOYMENT_POLICY_PATH = Path(P9_REUSE["deployment_policy_path"])
with open(P9_DEPLOYMENT_POLICY_PATH, "r", encoding="utf-8") as f:
    P9_DEPLOYMENT_POLICY = json.load(f)
MONITORED_COLS = sorted(P9_DEPLOYMENT_POLICY["monitored_features"])
P9_WEIGHTS = P9_DEPLOYMENT_POLICY["feature_weights"]["weights"]
P9_DIRECTIONS = P9_DEPLOYMENT_POLICY["feature_weights"]["directions"]
P9_MEANS = P9_DEPLOYMENT_POLICY["feature_weights"]["means"]
P9_STDS = P9_DEPLOYMENT_POLICY["feature_weights"]["stds"]
P9_CUT_LOW = P9_DEPLOYMENT_POLICY["cut_low"]
P9_CUT_HIGH = P9_DEPLOYMENT_POLICY["cut_high"]
COLLECTIONS_ELIGIBLE_STATES = P9_DEPLOYMENT_POLICY["collections_eligible_states"]
TREATMENT_TIER_RULE_NAMES = [t["name"] for t in P9_DEPLOYMENT_POLICY["treatment_tier_policy"]["tiers"]]

with open(Path(NB50_SUMMARY["policy_path"]), "r", encoding="utf-8") as f:
    _p9_business_policy = json.load(f)
STATE_NAMES = _p9_business_policy["reused_from_problem_8"]["state_names"]

P10_REUSE = CUSTOMER_INTELLIGENCE_POLICY["reused_from_problem_10"]
P10_WORKLIST_PATH = Path(P10_REUSE["worklist_path"])
if not P10_WORKLIST_PATH.exists():
    raise FileNotFoundError(f"{P10_WORKLIST_PATH} not found.\nFix: re-run Notebook 55 (Problem 10).")

for _p, _label in [(P9_MODEL_PATH, "Problem 9's persisted propensity model")]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found ({_label}).\nFix: re-run Notebook 52 (Problem 9).")

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
DETECTED_TOTAL_RAM_BYTES = PROJECT_CONFIG["resource_limits"]["total_ram_bytes_detected"]

P12_ROOT = PROJECT_ROOT / "Phase5_Customer_Business_Intelligence" / "Problem12_360_Customer_Intelligence"
if "customer_intelligence_validation_deployment" in PILLAR_DIRS:
    VALIDATION_DIR = PILLAR_DIRS["customer_intelligence_validation_deployment"]
else:
    VALIDATION_DIR = P12_ROOT / "validation_deployment"
    print(f"NOTE: 'customer_intelligence_validation_deployment' not in pillar_dirs -- using fallback: "
          f"{VALIDATION_DIR}")
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
API_SUBDIR = P12_ROOT / "src"
API_SUBDIR.mkdir(parents=True, exist_ok=True)
DOCS_SUBDIR = P12_ROOT / "docs"
DOCS_SUBDIR.mkdir(parents=True, exist_ok=True)

print(f"Loaded Notebook 62 policy from : {POLICY_PATH}")
print(f"Loaded Notebook 63 results from: {MODELING_RESULTS_PATH}")
print(f"Loaded Notebook 63 profile from: {PROFILE_PATH}")
print(f"Reported UNIFIED_RISK_SCORE ROC-AUC (Notebook 63): {REPORTED_UNIFIED_ROC_AUC:.4f} "
      f"(recommended_for_production: {REPORTED_RECOMMENDED_FOR_PRODUCTION})")
print(f"Validation artifacts will be written under: {VALIDATION_DIR}")
print("\n✅ Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS (PHASE 4/5
#            TIGHTENED 92%/92% CAP + TWO-TIER RAM GUARD, REUSED VERBATIM)
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

_PHASE5_CPU_FRACTION_CAP = 0.92
_PHASE5_RAM_FRACTION_CAP = 0.92
_historical_thread_count = PROJECT_CONFIG["resource_limits"]["warp_thread_count"]
_historical_max_ram_bytes = PROJECT_CONFIG["resource_limits"]["max_ram_bytes"]
WARP_THREAD_COUNT = min(_historical_thread_count, max(1, round(DETECTED_LOGICAL_CORES * _PHASE5_CPU_FRACTION_CAP)))
MAX_RAM_BYTES = min(_historical_max_ram_bytes, round(DETECTED_TOTAL_RAM_BYTES * _PHASE5_RAM_FRACTION_CAP))
os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import joblib
except ImportError:
    missing.append("joblib")
try:
    from sklearn.metrics import roc_auc_score
except ImportError:
    missing.append("scikit-learn")
try:
    from fastapi.testclient import TestClient
except ImportError:
    missing.append("fastapi")
if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


def _available_ram_gb() -> float:
    return psutil.virtual_memory().available / 1e9


# --- Two-tier (warn/hard-fail) RAM pre-flight guard, established after the
#     real 45-minute freeze the user hit on Notebook 52 -- this notebook
#     does one real streaming pass over the raw CSV (Section 4), the same
#     class of operation that caused that freeze. ---
_available_ram_gb_at_start = _available_ram_gb()
_comfortable_available_ram_gb = 0.50 * (MAX_RAM_BYTES / 1e9)
_min_required_available_ram_gb = 0.25 * (MAX_RAM_BYTES / 1e9)
if _available_ram_gb_at_start < _min_required_available_ram_gb:
    raise RuntimeError(
        f"Only {_available_ram_gb_at_start:.2f} GB of system RAM is available, below the "
        f"{_min_required_available_ram_gb:.2f} GB floor Section 4's reproduction needs. Close other "
        f"Jupyter kernels / applications, confirm with `psutil.virtual_memory().available / 1e9`, then "
        f"re-run this notebook from the top."
    )
if _available_ram_gb_at_start < _comfortable_available_ram_gb:
    print(f"⚠️  WARNING: only {_available_ram_gb_at_start:.2f} GB available "
          f"(comfortable margin {_comfortable_available_ram_gb:.2f} GB) -- proceeding.")
else:
    print(f"RAM pre-flight check passed: {_available_ram_gb_at_start:.2f} GB available.")

logger.info(f"Polars thread pool configured to {WARP_THREAD_COUNT}/{DETECTED_LOGICAL_CORES} threads "
            f"(Phase 5 tightened cap)")
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
print("\n✅ Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths")

_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break
if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n" + "\n".join(f"  - {c}" for c in _raw_candidates)
    )


def _resolve_pillar_file(filename: str, pillar_key: str, legacy_folder_name: str,
                          stored_path_str: str = None, min_size: int = 10_000) -> Path:
    """Same 3(+1)-candidate resolver every notebook in this platform uses."""
    _candidates = [
        PROJECT_ROOT / "Phase1_Foundation" / "Problem1_Credit_Scoring_PD_Prediction"
        / legacy_folder_name / filename,
    ]
    if pillar_key in PILLAR_DIRS:
        _candidates.append(PILLAR_DIRS[pillar_key] / filename)
    _candidates.append(PROJECT_ROOT / legacy_folder_name / filename)
    if stored_path_str:
        _candidates.append(Path(stored_path_str))
    for _c in _candidates:
        if _c.exists() and _c.stat().st_size > min_size:
            return _c
    raise FileNotFoundError(
        f"Could not resolve a real, non-trivial {filename}. Checked:\n"
        + "\n".join(f"  - {c}" for c in _candidates)
    )


TRAIN_SPLIT_PATH = _resolve_pillar_file(
    "train_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("train_split.csv"),
)
TEST_SPLIT_PATH = _resolve_pillar_file(
    "test_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("test_split.csv"),
)
print(f"Raw train_data.csv  : {RAW_TRAIN_DATA_PATH}")
print(f"train_split.csv     : {TRAIN_SPLIT_PATH}")
print(f"test_split.csv      : {TEST_SPLIT_PATH}")
print("\n✅ Section 3 complete.")


# =============================================================================
# SECTION 4: INDEPENDENT REPRODUCTION OF NOTEBOOK 63'S PIPELINE
# =============================================================================
_section("SECTION 4: Independent Reproduction of Notebook 63's Pipeline")

# --- Rebuilds Notebook 63's entire real pipeline from scratch, in a fresh
#     kernel: reloads Problem 10's real scored worklist, re-derives each
#     customer's real latest-statement severity/state via one real streaming
#     CSV pass, re-scores collections-eligible customers with Problem 9's
#     real persisted model, re-computes UNIFIED_RISK_SCORE, re-fits the
#     tertile cuts, and re-validates both hard-gating KPIs -- the same
#     independent-reproduction integrity check Notebooks 48/52/56 already
#     established for Problems 8/9/10. ---
BASE_DF = pl.read_parquet(P10_WORKLIST_PATH).rename({"TREND": "TREND_SEGMENT", "ACTION": "CREDIT_LINE_ACTION"})
print(f"Reloaded Problem 10's real worklist: {BASE_DF.height:,} customers")

_schema_overrides = {"customer_ID": pl.Utf8, "S_2": pl.Utf8}
for _c in MONITORED_COLS:
    _schema_overrides[_c] = pl.Float32
_inf_clean_exprs = [
    pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
    for c in MONITORED_COLS
]

print(f"Before latest-statement streaming pass -- RSS {_rss_gb():.2f} GB, "
      f"available RAM {_available_ram_gb():.2f} GB")
_t0 = time.time()
_latest_lf = (
    pl.scan_csv(RAW_TRAIN_DATA_PATH, schema_overrides=_schema_overrides)
    .with_row_index("_csv_row_order")
    .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d"))
    .with_columns(_inf_clean_exprs)
    .sort(["customer_ID", "S_2", "_csv_row_order"])
    .group_by("customer_ID", maintain_order=False)
    .agg([pl.col(c).last().alias(c) for c in MONITORED_COLS])
)
_wz_cols = []
for _c in MONITORED_COLS:
    _mean, _std, _w, _d = P9_MEANS[_c], P9_STDS[_c], P9_WEIGHTS[_c], P9_DIRECTIONS[_c]
    if _std > 0 and _w > 0:
        _expr = ((pl.col(_c) - _mean) / _std * _w * _d).fill_null(0.0).alias(f"_wz_{_c}")
    else:
        _expr = pl.lit(0.0).alias(f"_wz_{_c}")
    _wz_cols.append(_expr)
_state_expr = (
    pl.when(pl.col("SEVERITY_SCORE") <= P9_CUT_LOW).then(pl.lit(STATE_NAMES[0]))
    .when(pl.col("SEVERITY_SCORE") <= P9_CUT_HIGH).then(pl.lit(STATE_NAMES[1]))
    .otherwise(pl.lit(STATE_NAMES[2]))
    .alias("STATE")
)
LATEST_STATEMENT_DF = (
    _latest_lf
    .with_columns(_wz_cols)
    .with_columns(pl.sum_horizontal([f"_wz_{c}" for c in MONITORED_COLS]).alias("SEVERITY_SCORE"))
    .with_columns(_state_expr)
    .select(["customer_ID", "SEVERITY_SCORE", "STATE"] + MONITORED_COLS)
    .collect(engine="streaming")
)
LATEST_STATEMENT_DF = LATEST_STATEMENT_DF.with_columns(
    pl.col("STATE").is_in(COLLECTIONS_ELIGIBLE_STATES).alias("COLLECTIONS_ELIGIBLE")
)
print(f"Reproduced latest-statement severity/state in {time.time() - _t0:.1f}s -- "
      f"{LATEST_STATEMENT_DF.height:,} customers. RSS {_rss_gb():.2f} GB, "
      f"available RAM {_available_ram_gb():.2f} GB")

P9_MODEL = joblib.load(P9_MODEL_PATH)
_eligible_df = LATEST_STATEMENT_DF.filter(pl.col("COLLECTIONS_ELIGIBLE"))
_X_propensity = _eligible_df.select(MONITORED_COLS).to_numpy().astype(np.float32, copy=False)
_propensity_scores = P9_MODEL.predict_proba(_X_propensity)[:, 1] if len(_X_propensity) else np.array([])
PROPENSITY_DF = _eligible_df.select(["customer_ID", "SEVERITY_SCORE"]).with_columns(
    pl.Series("PROPENSITY_TO_CURE", _propensity_scores, dtype=pl.Float64)
)
if PROPENSITY_DF.height:
    _median_propensity = float(PROPENSITY_DF["PROPENSITY_TO_CURE"].median())
    _median_severity = float(PROPENSITY_DF["SEVERITY_SCORE"].median())
    _tier_expr = (
        pl.when((pl.col("PROPENSITY_TO_CURE") < _median_propensity) & (pl.col("SEVERITY_SCORE") >= _median_severity))
        .then(pl.lit(TREATMENT_TIER_RULE_NAMES[0]))
        .when(pl.col("PROPENSITY_TO_CURE") >= _median_propensity)
        .then(pl.lit(TREATMENT_TIER_RULE_NAMES[1]))
        .otherwise(pl.lit(TREATMENT_TIER_RULE_NAMES[2]))
        .alias("TREATMENT_TIER")
    )
    PROPENSITY_DF = PROPENSITY_DF.with_columns(_tier_expr).drop("SEVERITY_SCORE")
else:
    PROPENSITY_DF = PROPENSITY_DF.with_columns(pl.lit(None, dtype=pl.Utf8).alias("TREATMENT_TIER")).drop(
        "SEVERITY_SCORE")
print(f"Reproduced Problem 9 re-scoring: {PROPENSITY_DF.height:,} collections-eligible customers scored.")

UNIFIED_DF = (
    BASE_DF
    .join(LATEST_STATEMENT_DF.select(["customer_ID", "COLLECTIONS_ELIGIBLE"]), on="customer_ID", how="left")
    .join(PROPENSITY_DF, on="customer_ID", how="left")
    .with_columns(pl.col("COLLECTIONS_ELIGIBLE").fill_null(False))
)

_sw = UNIFIED_SCORE_WEIGHTS["static_pd_weight"]
_dw = UNIFIED_SCORE_WEIGHTS["dynamic_pd_weight"]
_cw = UNIFIED_SCORE_WEIGHTS["collections_adjustment_weight"]
_with_propensity_score = (
    (_sw * pl.col("STATIC_PD") + _dw * pl.col("DYNAMIC_PD") + _cw * (1.0 - pl.col("PROPENSITY_TO_CURE")))
    / (_sw + _dw + _cw)
)
_without_propensity_score = _sw * pl.col("STATIC_PD") + _dw * pl.col("DYNAMIC_PD")
UNIFIED_DF = UNIFIED_DF.with_columns(
    pl.when(pl.col("PROPENSITY_TO_CURE").is_not_null())
    .then(_with_propensity_score)
    .otherwise(_without_propensity_score)
    .alias("UNIFIED_RISK_SCORE")
)

TRAIN_IDS_DF = pl.read_csv(TRAIN_SPLIT_PATH, schema_overrides={"customer_ID": pl.Utf8}).select("customer_ID")
TEST_IDS_DF = pl.read_csv(TEST_SPLIT_PATH, schema_overrides={"customer_ID": pl.Utf8}).select("customer_ID")
TRAIN_DF = UNIFIED_DF.join(TRAIN_IDS_DF, on="customer_ID", how="inner")
HOLDOUT_DF = UNIFIED_DF.join(TEST_IDS_DF, on="customer_ID", how="inner")

_p_lo = UNIFIED_RISK_GRADE_CUT_PERCENTILES[0] / 100.0
_p_hi = UNIFIED_RISK_GRADE_CUT_PERCENTILES[1] / 100.0
REPRODUCED_GRADE_CUT_LOW = float(TRAIN_DF["UNIFIED_RISK_SCORE"].quantile(_p_lo))
REPRODUCED_GRADE_CUT_HIGH = float(TRAIN_DF["UNIFIED_RISK_SCORE"].quantile(_p_hi))


def _assign_grade(df: "pl.DataFrame") -> "pl.DataFrame":
    _grade_expr = (
        pl.when(pl.col("UNIFIED_RISK_SCORE") <= REPRODUCED_GRADE_CUT_LOW).then(pl.lit(UNIFIED_RISK_GRADE_NAMES[0]))
        .when(pl.col("UNIFIED_RISK_SCORE") <= REPRODUCED_GRADE_CUT_HIGH).then(pl.lit(UNIFIED_RISK_GRADE_NAMES[1]))
        .otherwise(pl.lit(UNIFIED_RISK_GRADE_NAMES[2]))
        .alias("UNIFIED_RISK_GRADE")
    )
    return df.with_columns(_grade_expr)


UNIFIED_DF = _assign_grade(UNIFIED_DF)
HOLDOUT_DF = _assign_grade(HOLDOUT_DF)

_y_holdout = HOLDOUT_DF["target"].to_numpy()
REPRODUCED_UNIFIED_ROC_AUC = float(roc_auc_score(_y_holdout, HOLDOUT_DF["UNIFIED_RISK_SCORE"].to_numpy()))
REPRODUCED_STATIC_PD_AUC = float(roc_auc_score(_y_holdout, HOLDOUT_DF["STATIC_PD"].to_numpy()))
REPRODUCED_DYNAMIC_PD_AUC = float(roc_auc_score(_y_holdout, HOLDOUT_DF["DYNAMIC_PD"].to_numpy()))
REPRODUCED_BEST_SINGLE_SIGNAL_AUC = max(REPRODUCED_STATIC_PD_AUC, REPRODUCED_DYNAMIC_PD_AUC)

print(f"\nReproduced UNIFIED_RISK_SCORE ROC-AUC : {REPRODUCED_UNIFIED_ROC_AUC:.6f}")
print(f"Reported   UNIFIED_RISK_SCORE ROC-AUC : {REPORTED_UNIFIED_ROC_AUC:.6f}")
_auc_diff = abs(REPRODUCED_UNIFIED_ROC_AUC - REPORTED_UNIFIED_ROC_AUC)
print(f"Reproduced grade cuts : low={REPRODUCED_GRADE_CUT_LOW:.6f}, high={REPRODUCED_GRADE_CUT_HIGH:.6f}")
print(f"Reported   grade cuts : low={REPORTED_GRADE_CUT_LOW:.6f}, high={REPORTED_GRADE_CUT_HIGH:.6f}")
_cut_diff = abs(REPRODUCED_GRADE_CUT_LOW - REPORTED_GRADE_CUT_LOW) + \
    abs(REPRODUCED_GRADE_CUT_HIGH - REPORTED_GRADE_CUT_HIGH)

# Same honest 1e-4 tolerance convention this platform's other reproduction checks use (Notebooks 48/52/56
# Section 4), for floating-point non-associativity across threaded reductions.
REPRODUCTION_PASSED = bool(_auc_diff < 1e-4 and _cut_diff < 1e-4)
print(f"\nReproduction diff (ROC-AUC): {_auc_diff:.8f}, (cuts): {_cut_diff:.8f} -- "
      f"{'PASS' if REPRODUCTION_PASSED else 'FAIL'}")
if not REPRODUCTION_PASSED:
    raise AssertionError(
        f"Notebook 63 did NOT reproduce (ROC-AUC diff {_auc_diff:.8f}, cut diff {_cut_diff:.8f}) -- do "
        f"not proceed to deployment until this is resolved."
    )

# --- Cross-check the persisted unified_customer_profile.parquet against this
#     fresh reproduction, for a real sample of holdout customers -- catches a
#     class of bug an aggregate-metric check alone cannot (a stale or
#     corrupted profile file on disk, even if the aggregate metrics above
#     happen to still match). ---
_persisted_profile = pl.read_parquet(PROFILE_PATH)
_sample_ids = HOLDOUT_DF.sort("customer_ID").head(200).select("customer_ID")
_repro_sample = HOLDOUT_DF.join(_sample_ids, on="customer_ID", how="inner").select(
    ["customer_ID", "UNIFIED_RISK_SCORE", "UNIFIED_RISK_GRADE", "TREATMENT_TIER"]
)
_persisted_sample = _persisted_profile.join(_sample_ids, on="customer_ID", how="inner").select(
    ["customer_ID", "UNIFIED_RISK_SCORE", "UNIFIED_RISK_GRADE", "TREATMENT_TIER"]
)
_compare = _repro_sample.join(
    _persisted_sample, on="customer_ID", how="inner", suffix="_persisted"
).with_columns((pl.col("UNIFIED_RISK_SCORE") - pl.col("UNIFIED_RISK_SCORE_persisted")).abs().alias("_urs_diff"))
_max_sample_diff = float(_compare["_urs_diff"].max()) if _compare.height > 0 else float("inf")
_grade_mismatches = int(
    (_compare["UNIFIED_RISK_GRADE"] != _compare["UNIFIED_RISK_GRADE_persisted"]).sum()
    + (_compare["TREATMENT_TIER"].fill_null("__null__")
       != _compare["TREATMENT_TIER_persisted"].fill_null("__null__")).sum()
)
PROFILE_VERIFIED = bool(
    _compare.height == _sample_ids.height and _max_sample_diff < 1e-4 and _grade_mismatches == 0
)
print(f"\nPersisted profile cross-check ({_compare.height} sampled holdout customers): "
      f"max UNIFIED_RISK_SCORE diff {_max_sample_diff:.8f}, grade/tier mismatches {_grade_mismatches} -- "
      f"{'PASS' if PROFILE_VERIFIED else 'FAIL'}")
if not PROFILE_VERIFIED:
    raise AssertionError(
        "The persisted unified profile does NOT match a fresh reproduction for the sampled customers -- do "
        "not deploy this artifact until this is resolved."
    )
print("\n✅ Section 4 complete.")


# =============================================================================
# SECTION 5: BOOTSTRAP CONFIDENCE INTERVAL -- COMPOSITE_NON_INFERIORITY GAP
# =============================================================================
_section("SECTION 5: Bootstrap Confidence Interval -- composite_non_inferiority Gap")

# --- profile_completeness is a deterministic null-count check on the entire
#     population, not a sampling-variable ratio/rate -- unlike
#     risk_level_monotonicity or trend_coherence in Notebooks 48/52/56, it
#     has no meaningful bootstrap distribution (resampling rows does not
#     change whether a column is null), so this notebook honestly bootstraps
#     only the KPI that is actually a sampling-variable statistic:
#     composite_non_inferiority's AUC gap. ---
_rng = np.random.default_rng(RANDOM_SEED)
_n_boot = 200
_n_holdout = HOLDOUT_DF.height
_holdout_y_arr = HOLDOUT_DF["target"].to_numpy()
_holdout_unified_arr = HOLDOUT_DF["UNIFIED_RISK_SCORE"].to_numpy()
_holdout_static_arr = HOLDOUT_DF["STATIC_PD"].to_numpy()
_holdout_dynamic_arr = HOLDOUT_DF["DYNAMIC_PD"].to_numpy()

_boot_gaps = np.empty(_n_boot, dtype=np.float64)
for _i in range(_n_boot):
    _idx = _rng.integers(0, _n_holdout, size=_n_holdout)
    _y_b = _holdout_y_arr[_idx]
    if _y_b.min() == _y_b.max():
        _boot_gaps[_i] = np.nan
        continue
    _unified_auc_b = roc_auc_score(_y_b, _holdout_unified_arr[_idx])
    _static_auc_b = roc_auc_score(_y_b, _holdout_static_arr[_idx])
    _dynamic_auc_b = roc_auc_score(_y_b, _holdout_dynamic_arr[_idx])
    _boot_gaps[_i] = _unified_auc_b - max(_static_auc_b, _dynamic_auc_b)

_boot_gaps_valid = _boot_gaps[~np.isnan(_boot_gaps)]
COMPOSITE_GAP_CI = [
    float(np.percentile(_boot_gaps_valid, 2.5)), float(np.percentile(_boot_gaps_valid, 97.5))
] if len(_boot_gaps_valid) > 0 else [float("nan"), float("nan")]
print(f"Bootstrap 95% CI on composite_non_inferiority gap (UNIFIED_AUC - best single-signal AUC) "
      f"({len(_boot_gaps_valid)} valid resamples of {_n_boot}): "
      f"[{COMPOSITE_GAP_CI[0]:.4f}, {COMPOSITE_GAP_CI[1]:.4f}]")

COMPOSITE_NON_INFERIORITY_CI_PASSED = bool(COMPOSITE_GAP_CI[0] > -REPORTED_COMPOSITE_TOLERANCE)
print(f"\ncomposite_non_inferiority CI lower bound > -{REPORTED_COMPOSITE_TOLERANCE} (gap CI stays within "
      f"the tolerated band): {COMPOSITE_NON_INFERIORITY_CI_PASSED}")
print("\n✅ Section 5 complete.")


# =============================================================================
# SECTION 6: HONEST LIMITATION -- DEPLOYMENT SCOPE & ASSUMPTIONS
# =============================================================================
_section("SECTION 6: Honest Limitation -- Deployment Scope & Assumptions")

MEETS_KPI_WITH_CI = bool(COMPOSITE_NON_INFERIORITY_CI_PASSED)
print(
    "DEPLOYMENT SCOPE (honest): this service serves Problem 12's real precomputed unified customer profile -- "
    "a lookup by customer_ID, not a live-compute-from-inputs endpoint. The profile itself composes four real, "
    "already-validated upstream signals (Problem 1's static PD, Problem 6's dynamic PD, Problem 9's real "
    "propensity-to-cure re-scored on each collections-eligible customer's own real LATEST statement, and "
    "Problem 10's real risk-level/trend/action worklist) into one UNIFIED_RISK_SCORE and UNIFIED_RISK_GRADE. "
    "It does NOT itself execute any action -- the grade and treatment tier are business-rule recommendations "
    "for a human or a downstream system to act on, not a claim that any specific action has been proven to "
    "change a real financial outcome.\n\n"
    "The propensity-to-cure component is honestly null for any customer whose latest statement is not in a "
    "collections-eligible state (Problem 9's model was never trained to score those customers) -- the service "
    "surfaces this null plainly rather than imputing a value.\n\n"
    f"RECOMMENDED FOR PRODUCTION: {MEETS_KPI_WITH_CI} (the composite_non_inferiority KPI's 95% bootstrap CI "
    f"{'holds' if MEETS_KPI_WITH_CI else 'does NOT hold'} on the real HOLDOUT split; profile_completeness "
    f"{'passed' if MODELING_RESULTS['kpi_results']['profile_completeness']['passed'] else 'did NOT pass'} "
    f"as a deterministic population-wide check in Notebook 63)."
)
print("\n✅ Section 6 complete.")


# =============================================================================
# SECTION 7: PERSIST DEPLOYMENT POLICY ARTIFACT
# =============================================================================
_section("SECTION 7: Persist Deployment Policy Artifact")

CUSTOMER_INTELLIGENCE_DEPLOYMENT_POLICY = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "unified_risk_grade_names": UNIFIED_RISK_GRADE_NAMES,
    "unified_score_weights": UNIFIED_SCORE_WEIGHTS,
    "unified_grade_cut_low": REPRODUCED_GRADE_CUT_LOW,
    "unified_grade_cut_high": REPRODUCED_GRADE_CUT_HIGH,
    "reported_unified_roc_auc": REPORTED_UNIFIED_ROC_AUC,
    "reproduced_unified_roc_auc": REPRODUCED_UNIFIED_ROC_AUC,
    "reproduction_passed": REPRODUCTION_PASSED,
    "profile_verified": PROFILE_VERIFIED,
    "composite_non_inferiority_gap_ci_95": COMPOSITE_GAP_CI,
    "composite_non_inferiority_ci_passed": COMPOSITE_NON_INFERIORITY_CI_PASSED,
    "profile_completeness_passed": MODELING_RESULTS["kpi_results"]["profile_completeness"]["passed"],
    "meets_kpi_with_ci": MEETS_KPI_WITH_CI,
    "recommended_for_production": bool(
        MEETS_KPI_WITH_CI and REPRODUCTION_PASSED and PROFILE_VERIFIED
        and MODELING_RESULTS["kpi_results"]["profile_completeness"]["passed"]
    ),
    "collections_eligible_states": COLLECTIONS_ELIGIBLE_STATES,
    "treatment_tier_rule_names": TREATMENT_TIER_RULE_NAMES,
    "profile_path": str(PROFILE_PATH),
    "random_seed": RANDOM_SEED,
}
deployment_policy_path = DOCS_SUBDIR / "customer_intelligence_deployment_policy.json"
with open(deployment_policy_path, "w", encoding="utf-8") as f:
    json.dump(CUSTOMER_INTELLIGENCE_DEPLOYMENT_POLICY, f, indent=2)
print(f"Wrote: {deployment_policy_path}")
print("\n✅ Section 7 complete.")


# =============================================================================
# SECTION 8: GENERATE customer_intelligence_lookup_service.py -- REAL,
#            RUNNABLE FASTAPI LOOKUP SERVICE WITH AUTH + EXPLAINABILITY
# =============================================================================
_section("SECTION 8: Generate customer_intelligence_lookup_service.py")

# --- Architecture note: Problem 12's deliverable is fundamentally a
#     PRECOMPUTED lookup artifact (unified_customer_profile.parquet), not a
#     live-compute-from-inputs model -- unlike Notebook 56's /recommend
#     endpoint (which composes two live inputs), this service loads the real
#     persisted profile once at startup and serves it by customer_ID. This
#     is the correct architecture for this specific deliverable: recomputing
#     the full four-signal composite per request would mean re-implementing
#     Problems 1/6/9/10's entire pipelines a second time inside a fifth
#     service, the exact duplication anti-pattern this platform has
#     repeatedly flagged (Notebook 46 Section 4, Notebook 50 Section 4,
#     Notebook 56 Section 8).
#
#     Explainability: deterministic rule narration (which cut thresholds the
#     real UNIFIED_RISK_SCORE fell into, and the real weighted contribution
#     of each of the up to three active input signals) -- the same
#     per-model-type choice this platform's established table already uses
#     for linear/weighted-composite scores (Problem 4/8's exact-linear-term
#     decomposition), since UNIFIED_RISK_SCORE is itself an explicit weighted
#     sum, not a trained classifier requiring occlusion-based estimation. ---
_policy_path_str = str(deployment_policy_path)
_profile_path_str = str(PROFILE_PATH)

CUSTOMER_INTELLIGENCE_SERVICE_TEMPLATE = "\n".join([
    "# AMEX Enterprise Credit Risk Platform -- 360 Degree Customer Intelligence Lookup API.",
    "# Auto-generated by 64_customer_intelligence_validation_deployment.ipynb.",
    "# Serves each customer's real precomputed UNIFIED_RISK_SCORE / UNIFIED_RISK_GRADE profile, composed",
    "# from Problem 1 (static PD), Problem 6 (dynamic PD), Problem 9 (propensity-to-cure, collections-",
    "# eligible customers only), and Problem 10 (risk-level/trend/action worklist).",
    "# Every endpoint except /health requires a valid X-API-Key header (see .env.example).",
    "# Run with:",
    "#     uvicorn customer_intelligence_lookup_service:app --host 0.0.0.0 --port 8012",
    "import json",
    "import logging",
    "import os",
    "import secrets",
    "from pathlib import Path",
    "from typing import List, Optional",
    "",
    "import polars as pl",
    "from fastapi import Depends, FastAPI, HTTPException, Security",
    "from fastapi.security import APIKeyHeader",
    "from pydantic import BaseModel",
    "",
    "_auth_logger = logging.getLogger(__name__ + \".auth\")",
    "_DEV_DEFAULT_API_KEY = \"dev-only-CHANGE-ME-before-deploying\"",
    "_api_key_header = APIKeyHeader(name=\"X-API-Key\", auto_error=False)",
    "",
    "",
    "def _configured_api_key() -> str:",
    "    key = os.environ.get(\"API_KEY\")",
    "    if not key:",
    "        _auth_logger.warning(",
    "            \"API_KEY is not set -- falling back to the published dev-only default. Set API_KEY \"",
    "            \"before deploying this service anywhere reachable by anyone but you.\"",
    "        )",
    "        return _DEV_DEFAULT_API_KEY",
    "    return key",
    "",
    "",
    "def require_api_key(presented: str = Security(_api_key_header)) -> str:",
    "    expected = _configured_api_key()",
    "    if not presented or not secrets.compare_digest(presented, expected):",
    "        raise HTTPException(status_code=401, detail=\"Missing or invalid X-API-Key header.\")",
    "    return presented",
    "",
    "",
    "POLICY_PATH = Path(os.environ.get(\"AMEX_P12_POLICY_PATH\", r\"__POLICY_PATH_TOKEN__\"))",
    "PROFILE_PATH = Path(os.environ.get(\"AMEX_P12_PROFILE_PATH\", r\"__PROFILE_PATH_TOKEN__\"))",
    "with open(POLICY_PATH, \"r\", encoding=\"utf-8\") as _f:",
    "    _POLICY = json.load(_f)",
    "",
    "UNIFIED_RISK_GRADE_NAMES = _POLICY[\"unified_risk_grade_names\"]",
    "UNIFIED_SCORE_WEIGHTS = _POLICY[\"unified_score_weights\"]",
    "GRADE_CUT_LOW = _POLICY[\"unified_grade_cut_low\"]",
    "GRADE_CUT_HIGH = _POLICY[\"unified_grade_cut_high\"]",
    "RECOMMENDED_FOR_PRODUCTION = _POLICY[\"recommended_for_production\"]",
    "",
    "_PROFILE_DF = pl.read_parquet(PROFILE_PATH)",
    "_PROFILE_INDEX = {row[\"customer_ID\"]: row for row in _PROFILE_DF.iter_rows(named=True)}",
    "",
    "",
    "class ProfileResponse(BaseModel):",
    "    customer_id: str",
    "    static_pd: float",
    "    dynamic_pd: float",
    "    pd_trend: float",
    "    risk_level: str",
    "    trend_segment: str",
    "    credit_line_action: str",
    "    collections_eligible: bool",
    "    propensity_to_cure: Optional[float] = None",
    "    treatment_tier: Optional[str] = None",
    "    unified_risk_score: float",
    "    unified_risk_grade: str",
    "    rationale: str",
    "    reasoning: List[str] = []",
    "    recommended_for_production: bool = RECOMMENDED_FOR_PRODUCTION",
    "",
    "",
    "app = FastAPI(",
    "    title=\"AMEX Enterprise Credit Risk Platform -- 360 Degree Customer Intelligence Lookup API\",",
    "    description=\"Serves each real customer's precomputed unified risk profile, composed from four real, \"",
    "                \"already-validated upstream signals. Every endpoint except /health requires a valid \"",
    "                \"X-API-Key header.\",",
    "    version=\"1.0.0\",",
    ")",
    "",
    "",
    "@app.get(\"/health\")",
    "def health():",
    "    return {\"status\": \"ok\", \"profiles_loaded\": len(_PROFILE_INDEX)}",
    "",
    "",
    "@app.get(\"/policy-info\", dependencies=[Depends(require_api_key)])",
    "def policy_info():",
    "    return {",
    "        \"unified_risk_grade_names\": UNIFIED_RISK_GRADE_NAMES,",
    "        \"unified_score_weights\": UNIFIED_SCORE_WEIGHTS,",
    "        \"unified_grade_cuts\": [GRADE_CUT_LOW, GRADE_CUT_HIGH],",
    "        \"recommended_for_production\": RECOMMENDED_FOR_PRODUCTION,",
    "        \"profiles_loaded\": len(_PROFILE_INDEX),",
    "    }",
    "",
    "",
    "@app.get(\"/profile/{customer_id}\", response_model=ProfileResponse, "
    "dependencies=[Depends(require_api_key)])",
    "def get_profile(customer_id: str):",
    "    row = _PROFILE_INDEX.get(customer_id)",
    "    if row is None:",
    "        raise HTTPException(status_code=404, detail=f\"No unified profile found for "
    "customer_id={customer_id!r}.\")",
    "    sw = UNIFIED_SCORE_WEIGHTS[\"static_pd_weight\"]",
    "    dw = UNIFIED_SCORE_WEIGHTS[\"dynamic_pd_weight\"]",
    "    cw = UNIFIED_SCORE_WEIGHTS[\"collections_adjustment_weight\"]",
    "    reasoning = [",
    "        f\"static_pd={row['STATIC_PD']:.4f} x weight {sw} -> {sw * row['STATIC_PD']:.4f}\",",
    "        f\"dynamic_pd={row['DYNAMIC_PD']:.4f} x weight {dw} -> {dw * row['DYNAMIC_PD']:.4f}\",",
    "    ]",
    "    if row.get(\"PROPENSITY_TO_CURE\") is not None:",
    "        reasoning.append(",
    "            f\"propensity_to_cure={row['PROPENSITY_TO_CURE']:.4f} (collections-eligible) x weight {cw} \"",
    "            f\"-> {cw * (1.0 - row['PROPENSITY_TO_CURE']):.4f} (as 1 - propensity), renormalized\"",
    "        )",
    "    else:",
    "        reasoning.append(\"propensity_to_cure=null (customer is not currently collections-eligible; \"",
    "                         \"the collections adjustment term is honestly excluded, not imputed)\")",
    "    reasoning.append(",
    "        f\"unified_risk_score={row['UNIFIED_RISK_SCORE']:.4f} -> {row['UNIFIED_RISK_GRADE']} \"",
    "        f\"(cuts: <= {GRADE_CUT_LOW:.4f} {UNIFIED_RISK_GRADE_NAMES[0]}, \"",
    "        f\"<= {GRADE_CUT_HIGH:.4f} {UNIFIED_RISK_GRADE_NAMES[1]}, else {UNIFIED_RISK_GRADE_NAMES[2]})\"",
    "    )",
    "    return ProfileResponse(",
    "        customer_id=row[\"customer_ID\"], static_pd=row[\"STATIC_PD\"], dynamic_pd=row[\"DYNAMIC_PD\"],",
    "        pd_trend=row[\"PD_TREND\"], risk_level=row[\"RISK_LEVEL\"], trend_segment=row[\"TREND_SEGMENT\"],",
    "        credit_line_action=row[\"CREDIT_LINE_ACTION\"], collections_eligible=row[\"COLLECTIONS_ELIGIBLE\"],",
    "        propensity_to_cure=row.get(\"PROPENSITY_TO_CURE\"), treatment_tier=row.get(\"TREATMENT_TIER\"),",
    "        unified_risk_score=row[\"UNIFIED_RISK_SCORE\"], unified_risk_grade=row[\"UNIFIED_RISK_GRADE\"],",
    "        rationale=f\"Composite of static PD ({sw:.0%}), dynamic PD ({dw:.0%})\"",
    "                  + (f\", and collections propensity ({cw:.0%})\" if row.get(\"PROPENSITY_TO_CURE\") "
    "is not None else \" (collections term not applicable)\") + \".\",",
    "        reasoning=reasoning,",
    "    )",
    "",
])
CUSTOMER_INTELLIGENCE_SERVICE_SOURCE = (
    CUSTOMER_INTELLIGENCE_SERVICE_TEMPLATE
    .replace("__POLICY_PATH_TOKEN__", _policy_path_str)
    .replace("__PROFILE_PATH_TOKEN__", _profile_path_str)
)

service_py_path = API_SUBDIR / "customer_intelligence_lookup_service.py"
with open(service_py_path, "w", encoding="utf-8") as f:
    f.write(CUSTOMER_INTELLIGENCE_SERVICE_SOURCE)
compile(CUSTOMER_INTELLIGENCE_SERVICE_SOURCE, str(service_py_path), "exec")
print(f"Generated {len(CUSTOMER_INTELLIGENCE_SERVICE_SOURCE.splitlines())} lines, syntax-checked OK.")
print(f"Saved -> {service_py_path}")
print("\n✅ Section 8 complete.")


# =============================================================================
# SECTION 9: LIVE SELF-TEST -- IMPORT THE GENERATED SERVICE & DRIVE IT WITH
#            REAL HOLDOUT CUSTOMERS' ACTUAL PROFILES
# =============================================================================
_section("SECTION 9: Live Self-Test -- Import the Generated Service & Drive It")

os.environ["AMEX_P12_POLICY_PATH"] = str(deployment_policy_path)
os.environ["AMEX_P12_PROFILE_PATH"] = str(PROFILE_PATH)
_TEST_API_KEY = "pytest-only-test-key"
os.environ["API_KEY"] = _TEST_API_KEY
_spec = importlib.util.spec_from_file_location("amex_customer_intelligence_service", str(service_py_path))
_service_module = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_service_module)
client = TestClient(_service_module.app)
_auth_headers = {"X-API-Key": _TEST_API_KEY}

_health_resp = client.get("/health")
assert _health_resp.status_code == 200
print(f"GET /health              (no key)   -> {_health_resp.status_code}  {_health_resp.json()}")

_unauth_resp = client.get("/policy-info")
assert _unauth_resp.status_code == 401
print(f"GET /policy-info         (no key, should reject) -> {_unauth_resp.status_code}")

_info_resp = client.get("/policy-info", headers=_auth_headers)
assert _info_resp.status_code == 200
print(f"GET /policy-info         (with key) -> {_info_resp.status_code}  "
      f"recommended_for_production={_info_resp.json()['recommended_for_production']}")

# Real end-to-end check against 3 real holdout customers: one with a real collections propensity
# score, one without, spanning different real grades.
_sample_with_propensity = HOLDOUT_DF.filter(pl.col("COLLECTIONS_ELIGIBLE")).sort("UNIFIED_RISK_SCORE")
_sample_without_propensity = HOLDOUT_DF.filter(~pl.col("COLLECTIONS_ELIGIBLE")).sort("UNIFIED_RISK_SCORE")
_sample_customers = []
if _sample_with_propensity.height:
    _sample_customers.append(_sample_with_propensity.row(_sample_with_propensity.height // 2, named=True))
if _sample_without_propensity.height:
    _sample_customers.append(_sample_without_propensity.row(_sample_without_propensity.height // 2, named=True))
_sample_customers.append(HOLDOUT_DF.sort("UNIFIED_RISK_SCORE", descending=True).row(0, named=True))

API_SELF_TEST_ROWS_PASSED = []
for _row in _sample_customers:
    _resp = client.get(f"/profile/{_row['customer_ID']}", headers=_auth_headers)
    assert _resp.status_code == 200, f"/profile returned {_resp.status_code}: {_resp.text}"
    _result = _resp.json()
    _row_passed = (
        _result["unified_risk_grade"] == _row["UNIFIED_RISK_GRADE"]
        and abs(_result["unified_risk_score"] - _row["UNIFIED_RISK_SCORE"]) < 1e-4
    )
    API_SELF_TEST_ROWS_PASSED.append(_row_passed)
    print(f"GET /profile/{_row['customer_ID']}: API grade={_result['unified_risk_grade']} "
          f"(expected {_row['UNIFIED_RISK_GRADE']}), score={_result['unified_risk_score']:.4f} "
          f"(expected {_row['UNIFIED_RISK_SCORE']:.4f}) -- {'PASS' if _row_passed else 'FAIL'}")

_unauth_profile_resp = client.get(f"/profile/{_sample_customers[0]['customer_ID']}")
assert _unauth_profile_resp.status_code == 401, "/profile without a key should be rejected"
print(f"GET /profile/...          (no key, should reject) -> {_unauth_profile_resp.status_code}")

_missing_resp = client.get("/profile/CUSTOMER_ID_THAT_DOES_NOT_EXIST", headers=_auth_headers)
assert _missing_resp.status_code == 404, "/profile for an unknown customer should 404"
print(f"GET /profile/<unknown>    (should 404) -> {_missing_resp.status_code}")

API_SELF_TEST_PASSED = bool(all(API_SELF_TEST_ROWS_PASSED))
if not API_SELF_TEST_PASSED:
    raise RuntimeError("Notebook 64's API self-test FAILED -- see checks above. Not safe to proceed.")
print("\n✅ Section 9 complete -- auth rejects unkeyed calls, all sampled real holdout customers' profiles "
      "match direct computation, an unknown customer_id correctly 404s.")


# =============================================================================
# SECTION 10: GENERATE .env.example & requirements-api.txt
# =============================================================================
_section("SECTION 10: Generate .env.example & requirements-api.txt")

_env_example = "\n".join([
    "# Copy to .env and fill in real values before deploying.",
    "API_KEY=dev-only-CHANGE-ME-before-deploying",
    f"AMEX_P12_POLICY_PATH={deployment_policy_path}",
    f"AMEX_P12_PROFILE_PATH={PROFILE_PATH}",
    "",
])
(API_SUBDIR / ".env.example").write_text(_env_example, encoding="utf-8")
_requirements_api = "\n".join(["fastapi>=0.110", "uvicorn>=0.29", "pydantic>=2.0", "polars>=0.20", ""])
(API_SUBDIR / "requirements-api.txt").write_text(_requirements_api, encoding="utf-8")
print(f"Wrote: {API_SUBDIR / '.env.example'}")
print(f"Wrote: {API_SUBDIR / 'requirements-api.txt'}")
print("\n✅ Section 10 complete.")


# =============================================================================
# SECTION 11: VERIFICATION -- INTEGRITY CHECKS
# =============================================================================
_section("SECTION 11: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Deployment policy file was written", deployment_policy_path.exists())
_all_checks_passed &= _check("Service file was written and syntax-checked", service_py_path.exists())
_all_checks_passed &= _check("Reproduction matches Notebook 63's reported values (ROC-AUC + cuts)",
                              REPRODUCTION_PASSED)
_all_checks_passed &= _check("Persisted unified profile verified against a fresh reproduction sample",
                              PROFILE_VERIFIED)
_all_checks_passed &= _check("Bootstrap composite_non_inferiority gap CI is well-formed (lower <= point <= upper)",
                              COMPOSITE_GAP_CI[0] <= (REPRODUCED_UNIFIED_ROC_AUC - REPRODUCED_BEST_SINGLE_SIGNAL_AUC)
                              <= COMPOSITE_GAP_CI[1])
_all_checks_passed &= _check("API self-test passed on all sampled real holdout customers", API_SELF_TEST_PASSED)
_all_checks_passed &= _check("Reproduced grade cuts are correctly ordered (low < high)",
                              REPRODUCED_GRADE_CUT_LOW < REPRODUCED_GRADE_CUT_HIGH)
_expected_files = [deployment_policy_path, service_py_path,
                   API_SUBDIR / ".env.example", API_SUBDIR / "requirements-api.txt"]
for _fp in _expected_files:
    _all_checks_passed &= _check(f"{_fp.name} exists and is non-empty", _fp.exists() and _fp.stat().st_size > 0)

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n✅ Section 11 complete -- all checks passed.")


# =============================================================================
# SECTION 12: WRITE NOTEBOOK 64 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 12: Write Notebook 64 Summary Artifact")

NB64_SUMMARY = {
    "notebook": "64_customer_intelligence_validation_deployment.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "deployment_policy_path": str(deployment_policy_path),
    "service_py_path": str(service_py_path),
    "reproduction_passed": REPRODUCTION_PASSED,
    "profile_verified": PROFILE_VERIFIED,
    "reproduced_unified_roc_auc": REPRODUCED_UNIFIED_ROC_AUC,
    "composite_non_inferiority_gap_ci_95": COMPOSITE_GAP_CI,
    "meets_kpi_with_ci": MEETS_KPI_WITH_CI,
    "recommended_for_production": CUSTOMER_INTELLIGENCE_DEPLOYMENT_POLICY["recommended_for_production"],
    "api_self_test_passed": API_SELF_TEST_PASSED,
    "random_seed": RANDOM_SEED,
}
NB64_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_64_summary.json"
with open(NB64_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB64_SUMMARY, f, indent=2)
print(f"Wrote: {NB64_SUMMARY_PATH}")

_section("NOTEBOOK 64 COMPLETE")
print(f"Reproduction passed                     : {REPRODUCTION_PASSED}")
print(f"Persisted profile verified              : {PROFILE_VERIFIED}")
print(f"Reproduced UNIFIED_RISK_SCORE ROC-AUC    : {REPRODUCED_UNIFIED_ROC_AUC:.4f}")
print(f"composite_non_inferiority (CI-adjusted)  : "
      f"{'PASS' if COMPOSITE_NON_INFERIORITY_CI_PASSED else 'FAIL'}")
print(f"RECOMMENDED_FOR_PRODUCTION               : "
      f"{CUSTOMER_INTELLIGENCE_DEPLOYMENT_POLICY['recommended_for_production']}")
print(f"API self-test                            : {'PASSED' if API_SELF_TEST_PASSED else 'FAILED'}")
print(f"Service written to: {service_py_path}")
print(
    "\nThis completes Problem 12 (360 Degree Customer Intelligence). "
    "Next: Problem 13 (Risk-Adjusted Profitability Modeling), Notebooks 66-69."
)
